In [10]:
%pip install --upgrade --quiet  pip
%pip install --upgrade --quiet  langchain-core
%pip install --upgrade --quiet  langchain-google-genai
%pip install --upgrade --quiet  langchain_community 


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


https://python.langchain.com/v0.1/docs/integrations/llms/

In [ ]:
import os
import textwrap
from IPython.display import display
from IPython.display import Markdown
from datetime import datetime
import numpy_financial as npf
import math
import pandas as pd
import numpy as np

if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = os.getenv('GEMINI_API_KEY')

def to_markdown(text):
  text = text.replace('•', '  *')
  return Markdown(textwrap.indent(text, '> ', predicate=lambda _: True))

# Embedding

## 觀察 Embedding 如何表示
https://cloud.google.com/vertex-ai/generative-ai/docs/embeddings/get-text-embeddings

In [10]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import InMemoryVectorStore
import json

embeddings_model = GoogleGenerativeAIEmbeddings(
    model="models/embedding-001"
)
vector = embeddings_model.embed_query("高雄大學")
print(f'vector維度: {len(vector)}')
print(vector[:10])  # This will print the first 10 elements of the vector

# Save the embeddings to a file
with open('embeddings.json',  'w', encoding='utf-8') as f:
    json.dump(vector, f, ensure_ascii=False, indent=4)

vector維度: 768
[0.042790789157152176, -0.06421862542629242, -0.02308056131005287, -0.004475655499845743, 0.03811584785580635, -0.003577709197998047, 0.016696317121386528, -0.01706259325146675, 0.010601178742945194, -0.002025854540988803]


## 相似度

In [ ]:
def cosine_similarity(vector_a, vector_b):
    """
    Calculate cosine similarity between two vectors.

    :param vector_a: First vector as a list or numpy array
    :param vector_b: Second vector as a list or numpy array
    :return: Cosine similarity (float)
    """
    # Convert to numpy arrays if not already
    vector_a = np.array(vector_a)
    vector_b = np.array(vector_b)

    # Compute the dot product and magnitudes
    dot_product = np.dot(vector_a, vector_b)
    magnitude_a = np.linalg.norm(vector_a)
    magnitude_b = np.linalg.norm(vector_b)

    # Avoid division by zero
    if magnitude_a == 0 or magnitude_b == 0:
        return 0.0

    return dot_product / (magnitude_a * magnitude_b)

In [ ]:
vector1 = embeddings_model.embed_query("apple")
vector2 = embeddings_model.embed_query("蘋果")
vector3 = embeddings_model.embed_query("orange")
vector4 = embeddings_model.embed_query("football")

In [7]:
similarity = cosine_similarity(vector1, vector3)
similarity

0.7025358394485549

# 問答
- invoke
- batch

In [ ]:
from google.generativeai.types import HarmBlockThreshold
from google.ai.generativelanguage_v1 import HarmCategory
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
        model='gemini-1.5-flash',
        temperature=1,
        safety_settings={
                HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT: HarmBlockThreshold.BLOCK_NONE,
                HarmCategory.HARM_CATEGORY_HARASSMENT: HarmBlockThreshold.BLOCK_NONE,
                HarmCategory.HARM_CATEGORY_HATE_SPEECH: HarmBlockThreshold.BLOCK_NONE,
                HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT : HarmBlockThreshold.BLOCK_NONE,
})



In [21]:
res = llm.invoke("你好")
print(res)
print(res.content)

content='你好！' additional_kwargs={} response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'safety_ratings': []} id='run-43c91d23-f6a3-4ff2-a9b1-c1d6e82f559d-0' usage_metadata={'input_tokens': 2, 'output_tokens': 3, 'total_tokens': 5, 'input_token_details': {'cache_read': 0}}
你好！


In [14]:
res_batch = llm.batch(["你好", "早安"])
res_batch

[AIMessage(content='你好！', additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'safety_ratings': []}, id='run-da052ef9-e1fd-4ca2-90e4-67215a39e97c-0', usage_metadata={'input_tokens': 2, 'output_tokens': 3, 'total_tokens': 5, 'input_token_details': {'cache_read': 0}}),
 AIMessage(content='早安！\n(Zǎo ān!)', additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'safety_ratings': []}, id='run-3af55547-5584-488c-af9f-843354b92b0f-0', usage_metadata={'input_tokens': 3, 'output_tokens': 12, 'total_tokens': 15, 'input_token_details': {'cache_read': 0}})]

In [20]:
for i in range(len(res_batch)):
    print(f"第{i}個回答: {res_batch[i].content}")
    print(res_batch[i].usage_metadata)

第0個回答: 你好！
{'input_tokens': 2, 'output_tokens': 3, 'total_tokens': 5, 'input_token_details': {'cache_read': 0}}
第1個回答: 早安！
(Zǎo ān!)
{'input_tokens': 3, 'output_tokens': 12, 'total_tokens': 15, 'input_token_details': {'cache_read': 0}}


# 範例

In [43]:
titles = []

with open("2330news_title.json", "r", encoding="utf-8") as f:
    news_title = json.load(f)
list_date = list(news_title.keys())
for d in list_date:
    print(f"日期: {d}")
    for t in news_title[d]:
        print(t.get('headline'))
        titles.append(t.get('headline'))
    print("-"*50)

日期: 20220601
台積電只排第45名！「這公司」年薪611萬最高
台積電、特斯拉生產線都攻下！高工學歷黑手老闆做減速機，如何搶下全球市占1成，翻身台灣隱形冠軍？
台積電高壓高效率 前員工：開會說「幾天後」，通常當天下午追進度
上市公司福利費用前10大！這家去年平均超過646萬 是台積電2倍多 
【鈔錢部署】台積電5年後EPS衝50元？ 這2檔趁反彈快賣
急拉過後，打回原形？
「浸潤式微影之父」林本堅的告白：在美國38年，為什麼我選擇在57歲那年，為了台積電回到台灣？
中科「台中園區擴建二期計畫」公聽 興農高球場會員爭取球證補償
兩岸晶片實力差多少？坂本幸雄：中芯最好產品已是8年前技術 「達不到討論競爭力的水平」
最強不是台積電！上市企業薪資排名 最狂公司「平均646萬」
--------------------------------------------------
日期: 20220602
台積電高雄設廠》那些沒機會討論的…
台積電、特斯拉生產線都攻下！高工學歷黑手老闆做減速機，如何搶下全球市占1成，翻身台灣隱形冠軍？
【鈔錢部署】台積電5年後EPS衝50元？ 這2檔趁反彈快賣
半導體年薪排行榜，這家超狂平均646萬！聯發科聯詠都超過500萬、台積電連「一半」都沒有…
實踐ESG精神！力行企業社會責任的十大製造業公民｜每日調查
期盼30年的「台美21世紀貿易倡議」，能給台灣哪些實質牛肉？
疫情兩年房市吹「南風」 這兩都房貸授信金額和人數超車台北
自由開講》內地化的香港 六四國殤被抹滅
魏寶生擔任富盛證券投顧副董 上任訂兩大營運目標
台積電擴廠欲徵收 寶山鄭家墓園申請文資審議未通過
--------------------------------------------------
日期: 20220603
半導體何以成為護國神山？台積電有哪些競爭對手？知名分析師帶你快速掌握-財經好讀-股票
台積電首曝美建廠1分鐘短片 外國網友盛讚amazing
台積電首曝美國建廠縮時短片 1分鐘震撼外國網友：Amazing！
台塑德山精密化學今視訊投料 產品IPA供應台積電
【英特爾的好幫手】退役軍人攜手200人台灣團隊 助半導體巨人鞏固AI版圖
6月3日產業動態最前線
外派到東南亞 10年當上廠長「真實薪資」曝光！網驚呆：太苦了
------------------

In [44]:
print(titles)

['台積電只排第45名！「這公司」年薪611萬最高', '台積電、特斯拉生產線都攻下！高工學歷黑手老闆做減速機，如何搶下全球市占1成，翻身台灣隱形冠軍？', '台積電高壓高效率 前員工：開會說「幾天後」，通常當天下午追進度', '上市公司福利費用前10大！這家去年平均超過646萬 是台積電2倍多 ', '【鈔錢部署】台積電5年後EPS衝50元？ 這2檔趁反彈快賣', '急拉過後，打回原形？', '「浸潤式微影之父」林本堅的告白：在美國38年，為什麼我選擇在57歲那年，為了台積電回到台灣？', '中科「台中園區擴建二期計畫」公聽 興農高球場會員爭取球證補償', '兩岸晶片實力差多少？坂本幸雄：中芯最好產品已是8年前技術 「達不到討論競爭力的水平」', '最強不是台積電！上市企業薪資排名 最狂公司「平均646萬」', '台積電高雄設廠》那些沒機會討論的…', '台積電、特斯拉生產線都攻下！高工學歷黑手老闆做減速機，如何搶下全球市占1成，翻身台灣隱形冠軍？', '【鈔錢部署】台積電5年後EPS衝50元？ 這2檔趁反彈快賣', '半導體年薪排行榜，這家超狂平均646萬！聯發科聯詠都超過500萬、台積電連「一半」都沒有…', '實踐ESG精神！力行企業社會責任的十大製造業公民｜每日調查', '期盼30年的「台美21世紀貿易倡議」，能給台灣哪些實質牛肉？', '疫情兩年房市吹「南風」 這兩都房貸授信金額和人數超車台北', '自由開講》內地化的香港 六四國殤被抹滅', '魏寶生擔任富盛證券投顧副董 上任訂兩大營運目標', '台積電擴廠欲徵收 寶山鄭家墓園申請文資審議未通過', '半導體何以成為護國神山？台積電有哪些競爭對手？知名分析師帶你快速掌握-財經好讀-股票', '台積電首曝美建廠1分鐘短片 外國網友盛讚amazing', '台積電首曝美國建廠縮時短片 1分鐘震撼外國網友：Amazing！', '台塑德山精密化學今視訊投料 產品IPA供應台積電', '【英特爾的好幫手】退役軍人攜手200人台灣團隊 助半導體巨人鞏固AI版圖', '6月3日產業動態最前線', '外派到東南亞 10年當上廠長「真實薪資」曝光！網驚呆：太苦了', '台積電美國 5 奈米廠 2024 年量產，建廠影片受關注', '讓「神操作」的人生揚眉又吐氣—鐘冠智勇敢跨出舒適圈 在「澳洲台積電」闖出一片天', '台

In [45]:
embeddings = []
for title in titles: 
    embd_model = GoogleGenerativeAIEmbeddings(
        model="models/embedding-001"
    )
    embd_vector = embd_model.embed_query(title)
    embeddings.extend([embd_vector])
print(embeddings)

[[0.05527857318520546, -0.02452155202627182, -0.04423009231686592, -0.010281476192176342, 0.047968581318855286, 0.008957309648394585, 0.014609713107347488, -0.03369539976119995, 0.01754794269800186, 0.006485679652541876, 0.032993417233228683, -0.023158635944128036, 0.02374226786196232, -0.04462360218167305, 0.009992416948080063, -0.047955602407455444, -0.0066658626310527325, -0.011987880803644657, -0.0038180651608854532, -0.008842675015330315, -0.02370060235261917, 0.02988860197365284, -0.0220232792198658, -0.013674875721335411, -0.00481497822329402, -0.04773906618356705, 0.02394985780119896, -0.07592378556728363, -0.00923085119575262, 0.061929598450660706, -0.07276105135679245, 0.039721954613924026, -0.04512900486588478, 0.0001977783686015755, 0.015577686950564384, -0.042694348841905594, -0.012715242803096771, 0.007924223318696022, 0.021250467747449875, 0.06747028231620789, -0.005690960679203272, -0.02941027842462063, -0.024296456947922707, -0.05350835248827934, 0.037459321320056915, 

In [46]:
for i in range(len(embeddings)):
    print(f"第{i}個標題/embedding:")
    print(f"{titles[i]}")
    print(f"{embeddings[i]}")
    print("")

第0個標題/embedding:
台積電只排第45名！「這公司」年薪611萬最高
[0.05527857318520546, -0.02452155202627182, -0.04423009231686592, -0.010281476192176342, 0.047968581318855286, 0.008957309648394585, 0.014609713107347488, -0.03369539976119995, 0.01754794269800186, 0.006485679652541876, 0.032993417233228683, -0.023158635944128036, 0.02374226786196232, -0.04462360218167305, 0.009992416948080063, -0.047955602407455444, -0.0066658626310527325, -0.011987880803644657, -0.0038180651608854532, -0.008842675015330315, -0.02370060235261917, 0.02988860197365284, -0.0220232792198658, -0.013674875721335411, -0.00481497822329402, -0.04773906618356705, 0.02394985780119896, -0.07592378556728363, -0.00923085119575262, 0.061929598450660706, -0.07276105135679245, 0.039721954613924026, -0.04512900486588478, 0.0001977783686015755, 0.015577686950564384, -0.042694348841905594, -0.012715242803096771, 0.007924223318696022, 0.021250467747449875, 0.06747028231620789, -0.005690960679203272, -0.02941027842462063, -0.024296456947922707, -0.0

In [53]:
from sklearn.cluster import KMeans
# 分群
def cluster_titles_kmeans(embeddings, n_clusters=5):
    kmeans = KMeans(n_clusters=n_clusters, random_state=42)
    labels = kmeans.fit_predict(embeddings)
    return kmeans, labels

# 分群
kmeans, labels = cluster_titles_kmeans(np.array(embeddings))
print(kmeans, labels)


KMeans(n_clusters=5, random_state=42) [2 2 2 0 2 2 2 4 2 0 4 2 2 0 2 2 4 4 4 4 2 4 2 4 4 0 1 4 4 4 0 0 4 2 4 2 4
 2 4 2 4 4 2 2 3 4 2 4 2 0 4 4 4 2 2 2 2 4 2 4 4 3 2 1 0 3 3 0 2 4 4 2 2 3
 4 4 0 4 2 2 4 4 2 2 3]


In [56]:
# 找出代表標題
def extract_representative_titles(titles, embeddings, kmeans, labels):
    clusters = {i: [] for i in range(kmeans.n_clusters)}
    for label, title, embedding in zip(labels, titles, embeddings):
        clusters[label].append((title, embedding))
    
    representative_titles = {}
    for label, items in clusters.items():
        cluster_center = kmeans.cluster_centers_[label]
        # 找出距離群中心最近的標題
        closest_title = min(
            items, key=lambda x: np.linalg.norm(np.array(x[1]) - cluster_center)
        )[0]
        representative_titles[label] = {
            "representative": closest_title,
            "titles": [item[0] for item in items]
        }
    return representative_titles

representative_titles = extract_representative_titles(titles, embeddings, kmeans, labels)

for label, cluster in representative_titles.items():
    news = f"{cluster['representative']}"
    for i, title in enumerate(cluster['titles']):
        if i > 3: # 只顯示代表標題以及前三個標題
            break
        news += f"  - {title}\n"

    print(news)

最強不是台積電！上市企業薪資排名 最狂公司「平均646萬」  - 上市公司福利費用前10大！這家去年平均超過646萬 是台積電2倍多 
  - 最強不是台積電！上市企業薪資排名 最狂公司「平均646萬」
  - 半導體年薪排行榜，這家超狂平均646萬！聯發科聯詠都超過500萬、台積電連「一半」都沒有…
  - 6月3日產業動態最前線

外派到東南亞 10年當上廠長「真實薪資」曝光！網驚呆：太苦了  - 外派到東南亞 10年當上廠長「真實薪資」曝光！網驚呆：太苦了
  - 台積電股東會懶人包來了 10大重點Q＆A一次看

在台積電工作十年，毅然辭去！瓦基：這輩子，一定有比「金飯碗」更重要的事  - 台積電只排第45名！「這公司」年薪611萬最高
  - 台積電、特斯拉生產線都攻下！高工學歷黑手老闆做減速機，如何搶下全球市占1成，翻身台灣隱形冠軍？
  - 台積電高壓高效率 前員工：開會說「幾天後」，通常當天下午追進度
  - 【鈔錢部署】台積電5年後EPS衝50元？ 這2檔趁反彈快賣

比M2晶片更高階 傳台積電下半年量產M2 Pro  - 蘋果：相同耗電下 M2效能遠優Intel十核心處理器
  - 蘋果發表M2處理器採台積電第二代5奈米- 財經要聞- 工商時報
  - 蘋果M2處理器 台積5奈米獨家代工
  - 傳蘋果M2 家族高階晶片M2 Pro 將採更先進的台積電3nm 製程， 2022 年末量產#Apple Silicon (178272)

【芯片荒】中國智庫陳文玲：一定要收復台灣 將台積電搶到中國手裡  - 中科「台中園區擴建二期計畫」公聽 興農高球場會員爭取球證補償
  - 台積電高雄設廠》那些沒機會討論的…
  - 疫情兩年房市吹「南風」 這兩都房貸授信金額和人數超車台北
  - 自由開講》內地化的香港 六四國殤被抹滅



In [61]:
import re

llm = ChatGoogleGenerativeAI(
        model='gemini-1.5-flash',
        temperature=1,
        safety_settings={
                HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT: HarmBlockThreshold.BLOCK_NONE,
                HarmCategory.HARM_CATEGORY_HARASSMENT: HarmBlockThreshold.BLOCK_NONE,
                HarmCategory.HARM_CATEGORY_HATE_SPEECH: HarmBlockThreshold.BLOCK_NONE,
                HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT : HarmBlockThreshold.BLOCK_NONE,
})

res_facotrs = llm.invoke(f"""
以下的新聞出現後隔日台積電的股價波動有關：
『{news}』

請依據上述的新聞列出5個會造成股價波動的可能原因，
以台灣中文回答""")

res_factors_json = llm.invoke(res_facotrs.content + """
{
    "1": "xxx：xxxxx"
    "2": 
    "3": 
    "4": 
    "5":
}
把這些因素轉換成這樣的格式，你只需要回答我轉換後的樣子就好""")

json_str = re.search(
    r'\{.*\}', res_factors_json.content, re.DOTALL).group()
json_data = json.loads(json_str)
with open("factors.json", 'w', encoding='utf-8') as f:
    json.dump(json_data, f, ensure_ascii=False, indent=4)

In [64]:
print(json_data)
for i, (k, v) in enumerate(json_data.items()):
    print(f"{i+1}. {v}")

{'1': '政治風險：中國智庫學者喊話『收復台灣，搶佔台積電』，加劇地緣政治風險，威脅台積電在台營運穩定性，恐導致股價下跌。', '2': '地緣政治風險：國際局勢緊張升溫，投資人避險情緒升高，拋售股票尋求更穩定投資標的，導致台積電股價下跌。', '3': '營運風險：市場對台積電高雄設廠及擴廠計畫的進度、效益存疑，可能存在未公開挑戰或風險，影響投資人信心，進而影響股價。', '4': '市場氛圍與國際情勢：整體市場下跌趨勢（受國際情勢如香港局勢、台灣房市等影響），台積電作為權值股難以倖免，股價可能隨之波動。', '5': '投資人情緒與解讀：投資人對新聞的解讀和反應差異可能造成股價波動，例如對中國智庫言論的過度反應。 '}
1. 政治風險：中國智庫學者喊話『收復台灣，搶佔台積電』，加劇地緣政治風險，威脅台積電在台營運穩定性，恐導致股價下跌。
2. 地緣政治風險：國際局勢緊張升溫，投資人避險情緒升高，拋售股票尋求更穩定投資標的，導致台積電股價下跌。
3. 營運風險：市場對台積電高雄設廠及擴廠計畫的進度、效益存疑，可能存在未公開挑戰或風險，影響投資人信心，進而影響股價。
4. 市場氛圍與國際情勢：整體市場下跌趨勢（受國際情勢如香港局勢、台灣房市等影響），台積電作為權值股難以倖免，股價可能隨之波動。
5. 投資人情緒與解讀：投資人對新聞的解讀和反應差異可能造成股價波動，例如對中國智庫言論的過度反應。 


In [69]:
value_list = json_data.values()
value_list

dict_values(['政治風險：中國智庫學者喊話『收復台灣，搶佔台積電』，加劇地緣政治風險，威脅台積電在台營運穩定性，恐導致股價下跌。', '地緣政治風險：國際局勢緊張升溫，投資人避險情緒升高，拋售股票尋求更穩定投資標的，導致台積電股價下跌。', '營運風險：市場對台積電高雄設廠及擴廠計畫的進度、效益存疑，可能存在未公開挑戰或風險，影響投資人信心，進而影響股價。', '市場氛圍與國際情勢：整體市場下跌趨勢（受國際情勢如香港局勢、台灣房市等影響），台積電作為權值股難以倖免，股價可能隨之波動。', '投資人情緒與解讀：投資人對新聞的解讀和反應差異可能造成股價波動，例如對中國智庫言論的過度反應。 '])

In [75]:
with open("2330news_title.json", "r", encoding="utf-8") as f:
    news_title = json.load(f)



list_date = list(news_title.keys())
for d in list_date[:2]:     # 只取前兩天測試
    print(f"日期: {d}")
    text_news = "today's news: \n"
    for news in news_title[d]:
        text_news += f"{news.get('headline')}\n"
    

    batch = [] # 用來同時問多個問題
    for factor in value_list:
        messages = [
            ("system", "你是一個厲害的金融領域專家"),
            ("human",
            f"""
            這是股價變動因素{factor}
            請思考以下新聞，是否會因為{factor}而使股票上漲或下跌？
            這是今天的新聞: {text_news}
            如果可能上漲請回答yes，如果可能下跌則回答no，如果無法判斷或無關則回答unknown。

            用以下格式回答，你只需要綜合所有的新聞給出一次的回答就好：
            判斷結果: #yes/#no/#unknown 請盡量不要回答unknown
            你的理由: **非常簡短**的用 1∼2 句話寫出來！
            """)
        ]
        print(messages)
        batch.append(messages)
    res = llm.batch(batch)
    
    for i, r in enumerate(res):
        print(f"第{i+1}個問題的回答:")
        print(r.content)
        print(r.usage_metadata)
        print("_" * 50)
    

日期: 20220601
[('system', '你是一個厲害的金融領域專家'), ('human', "\n            這是股價變動因素政治風險：中國智庫學者喊話『收復台灣，搶佔台積電』，加劇地緣政治風險，威脅台積電在台營運穩定性，恐導致股價下跌。\n            請思考以下新聞，是否會因為政治風險：中國智庫學者喊話『收復台灣，搶佔台積電』，加劇地緣政治風險，威脅台積電在台營運穩定性，恐導致股價下跌。而使股票上漲或下跌？\n            這是今天的新聞: today's news: \n台積電只排第45名！「這公司」年薪611萬最高\n台積電、特斯拉生產線都攻下！高工學歷黑手老闆做減速機，如何搶下全球市占1成，翻身台灣隱形冠軍？\n台積電高壓高效率 前員工：開會說「幾天後」，通常當天下午追進度\n上市公司福利費用前10大！這家去年平均超過646萬 是台積電2倍多 \n【鈔錢部署】台積電5年後EPS衝50元？ 這2檔趁反彈快賣\n急拉過後，打回原形？\n「浸潤式微影之父」林本堅的告白：在美國38年，為什麼我選擇在57歲那年，為了台積電回到台灣？\n中科「台中園區擴建二期計畫」公聽 興農高球場會員爭取球證補償\n兩岸晶片實力差多少？坂本幸雄：中芯最好產品已是8年前技術 「達不到討論競爭力的水平」\n最強不是台積電！上市企業薪資排名 最狂公司「平均646萬」\n\n            如果可能上漲請回答yes，如果可能下跌則回答no，如果無法判斷或無關則回答unknown。\n\n            用以下格式回答，你只需要綜合所有的新聞給出一次的回答就好：\n            判斷結果: #yes/#no/#unknown 請盡量不要回答unknown\n            你的理由: **非常簡短**的用 1∼2 句話寫出來！\n            ")]
[('system', '你是一個厲害的金融領域專家'), ('human', "\n            這是股價變動因素地緣政治風險：國際局勢緊張升溫，投資人避險情緒升高，拋售股票尋求更穩定投資標的，導致台積電股價下跌。\n            請思考以下新聞，是否會因為地緣政治風險：國際局勢緊張升溫，投資人避險情緒升高，拋售股票尋求更穩定投資標的，